In [1]:
import pandas as pd
import sqlite3

# load the cleaned training data
df_train = pd.read_csv("../data/train.csv")

# fix the missing value same as before
df_train['Arrival Delay in Minutes'] = df_train['Arrival Delay in Minutes'].fillna(
    df_train['Arrival Delay in Minutes'].median()
)

# create satisfaction binary column
df_train['is_dissatisfied'] = (
    df_train['satisfaction'] == 'neutral or dissatisfied'
).astype(int)

# connect to SQLite — this creates the database file automatically
conn = sqlite3.connect("../data/airline_satisfaction.db")

# load the dataframe into the database as a table
df_train.to_sql("passengers", conn, if_exists="replace", index=False)

print("Database created successfully")
print(f"Total rows loaded: {len(df_train)}")

Database created successfully
Total rows loaded: 103904


In [2]:
import os
print(os.getcwd())

C:\Users\likhi\OneDrive\Desktop\airline satisfaction project\notebook


In [3]:
def run_query(sql):
    return pd.read_sql_query(sql, conn)

In [4]:
run_query("""
    SELECT 
        Class,
        COUNT(*) as total_passengers,
        SUM(is_dissatisfied) as dissatisfied_count,
        ROUND(AVG(is_dissatisfied) * 100, 1) as dissatisfaction_rate
    FROM passengers
    GROUP BY Class
    ORDER BY dissatisfaction_rate DESC
""")

,Class,total_passengers,dissatisfied_count,dissatisfaction_rate
0,Eco,46745,38044,81.4
1,Eco Plus,7494,5650,75.4
2,Business,49665,15185,30.6


In [5]:
run_query("""
    SELECT 
        `Type of Travel`,
        COUNT(*) as total_passengers,
        SUM(is_dissatisfied) as dissatisfied_count,
        ROUND(AVG(is_dissatisfied) * 100, 1) as dissatisfaction_rate
    FROM passengers
    GROUP BY `Type of Travel`
    ORDER BY dissatisfaction_rate DESC
""")

,Type of Travel,total_passengers,dissatisfied_count,dissatisfaction_rate
0,Personal Travel,32249,28970,89.8
1,Business travel,71655,29909,41.7


In [6]:
run_query("""
    SELECT
        satisfaction,
        ROUND(AVG(`Inflight wifi service`), 2) as avg_wifi,
        ROUND(AVG(`Online boarding`), 2) as avg_online_boarding,
        ROUND(AVG(`Seat comfort`), 2) as avg_seat_comfort,
        ROUND(AVG(`Inflight entertainment`), 2) as avg_entertainment,
        ROUND(AVG(`Leg room service`), 2) as avg_legroom,
        ROUND(AVG(`On-board service`), 2) as avg_onboard_service,
        ROUND(AVG(`Cleanliness`), 2) as avg_cleanliness,
        ROUND(AVG(`Food and drink`), 2) as avg_food
    FROM passengers
    GROUP BY satisfaction
""")

,satisfaction,avg_wifi,avg_online_boarding,avg_seat_comfort,avg_entertainment,avg_legroom,avg_onboard_service,avg_cleanliness,avg_food
0,neutral or dissatisfied,2.40,2.66,3.04,2.89,2.99,3.02,2.94,2.96
1,satisfied,3.16,4.03,3.97,3.96,3.82,3.86,3.74,3.52


In [13]:
run_query("""
    WITH dissatisfied_counts AS (
        SELECT
            Class,
            COUNT(*) as total_passengers,
            SUM(is_dissatisfied) as dissatisfied_passengers
        FROM passengers
        GROUP BY Class
    )
    
    SELECT
        Class,
        total_passengers,
        dissatisfied_passengers,

        CASE 
            WHEN Class = 'Business' THEN 500
            WHEN Class = 'Eco Plus' THEN 250
            WHEN Class = 'Eco' THEN 150
        END as revenue_per_passenger,

        CASE 
            WHEN Class = 'Business' THEN dissatisfied_passengers * 500
            WHEN Class = 'Eco Plus' THEN dissatisfied_passengers * 250
            WHEN Class = 'Eco' THEN dissatisfied_passengers * 150
        END as revenue_at_risk,

        RANK() OVER (
            ORDER BY 
            CASE 
                WHEN Class = 'Business' THEN dissatisfied_passengers * 500
                WHEN Class = 'Eco Plus' THEN dissatisfied_passengers * 250
                WHEN Class = 'Eco' THEN dissatisfied_passengers * 150
            END DESC
        ) as risk_rank

    FROM dissatisfied_counts
    order by revenue_at_risk
""")

,Class,total_passengers,dissatisfied_passengers,revenue_per_passenger,revenue_at_risk,risk_rank
0,Eco Plus,7494,5650,250,1412500,3
1,Eco,46745,38044,150,5706600,2
2,Business,49665,15185,500,7592500,1


## BQ4 Findings — Revenue at Risk

| Class    | Dissatisfied Passengers | Revenue at Risk |
|----------|------------------------|-----------------|
| Business | 15,185                 | $7,592,500      |
| Eco      | 38,044                 | $5,706,600      |
| Eco Plus | 5,650                  | $1,412,500      |

Total revenue at risk: $14,711,600

Note: Revenue per passenger estimated using industry averages.
Business class ranks #1 in revenue at risk despite lowest 
dissatisfaction rate — each dissatisfied Business passenger 
represents $500 in lost revenue.

each Business passenger represents $500. 
    
Even a small percentage of unhappy Business passengers costs more than a large percentage of unhappy Economy passengers.